# Training notebook

This is a cleaned-up notebook used for training and testing individual RF, MLP and CNN models. This notebook has now been updated to train on R1a, R1b and R2 regions, holding out the R3 region as a validation check. This is a better metric to use for validation than the random sample that was done before, as it avoids the problem of pixels in the training set being correlated with those in the validation set.

This notebook does not load PCs, since it has now been established that this gives a reduction in performance.



In [ ]:
# IMPORTS + CONFIG

import pandas as pd
import numpy as np
import joblib
import time
import gc
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE = 42

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [2]:
# LOAD DATA

dfs = [pd.read_parquet(LOCAL_DIR / f'Merged\\{r.lower()}_combined.parquet') for r in REGIONS]
df_all = pd.concat(dfs, ignore_index=True)
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]

In [3]:
# STRATIFIED SAMPLING - USES QUINTILES BY DEFAULT

PER_BUCKET_TARGET = 20000


_, quintile_bins = pd.qcut(df_all['edge_distance'], q=5, retbins=True, duplicates='drop')
quintile_bins[0] = -0.1
quintile_bins[-1] = np.inf

df_all['quintile'] = pd.cut(
    df_all['edge_distance'],
    bins=quintile_bins,
    labels=[1, 2, 3, 4, 5]
).astype(int)

print('Population per (quintile x class):')
pop = df_all.groupby(['quintile', 'melt_label']).size().unstack(fill_value=0)
pop['melt_frac'] = (pop[1] / pop.sum(axis=1)).round(3)
print(pop)
print('\nQuintile boundaries used:', np.round(quintile_bins, 1).tolist())

sampled = []
for (q, m), group in df_all.groupby(['quintile', 'melt_label']):
    n_take = min(PER_BUCKET_TARGET, len(group))
    sampled.append(group.sample(n=n_take, random_state=RANDOM_STATE))
df_train_full = pd.concat(sampled, ignore_index=True)

print(f'\nTraining set: {len(df_train_full):,} samples '
      f'({df_train_full["melt_label"].mean()*100:.1f}% melt)')
print('\nSampled per (quintile x class):')
print(df_train_full.groupby(['quintile', 'melt_label']).size().unstack(fill_value=0))
print('\nEdge-distance mean by class:')
print(df_train_full.groupby('melt_label')['edge_distance'].mean().round(1))

Population per (quintile x class):
melt_label        0        1  melt_frac
quintile                               
1            979372  2069160      0.679
2           2365708   682816      0.224
3           2873996   174534      0.057
4           3010543    37983      0.012
5           3041336     7191      0.002

Quintile boundaries used: [-0.1, 37.8, 87.8, 164.0, 301.8, inf]

Training set: 187,191 samples (46.6% melt)

Sampled per (quintile x class):
melt_label      0      1
quintile                
1           20000  20000
2           20000  20000
3           20000  20000
4           20000  20000
5           20000   7191

Edge-distance mean by class:
melt_label
0    191.100006
1    122.699997
Name: edge_distance, dtype: float32


In [3]:
# CLASS BALANCED SAMPLING (NO STRATIFICATION)

N_PER_CLASS = 250000

melt_sample = df_all[df_all['melt_label'] == 1].sample(
    n=min(N_PER_CLASS, (df_all['melt_label'] == 1).sum()),
    random_state=RANDOM_STATE)
nonmelt_sample = df_all[df_all['melt_label'] == 0].sample(
    n=N_PER_CLASS, random_state=RANDOM_STATE)
df_train_full = pd.concat([melt_sample, nonmelt_sample], ignore_index=True)
print(f'Class-balanced sample: {len(df_train_full):,} pixels, '
      f'{df_train_full["melt_label"].mean()*100:.1f}% melt')

Class-balanced sample: 500,000 pixels, 50.0% melt


In [4]:
# SELECT FEATURES + CONSTRUCT TRAIN/VAL SETS

# NOTE: FOR ALL MLP MODELS, FEATURE ORDER MATTERS. USE [elevation, edge_distance] IN THAT ORDER

FEATURE_COLS = ['elevation', 'edge_distance', 'aspect', 'slope']
print(f'Using {len(FEATURE_COLS)} features.')

X = df_train_full[FEATURE_COLS].values
y = df_train_full['melt_label'].values
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

Using 4 features.


In [5]:
# RANDOM FOREST MODEL TRAINING + EVALUATION

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')
print('Training...')

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)

def cm_errors(cm):
    ice  = 1 - cm[0, 0] / cm[0].sum()
    melt = 1 - cm[1, 1] / cm[1].sum()
    avg  = (cm[0, 1] + cm[1, 0]) / cm.sum()
    return ice, melt, avg


# OOB error (unbiased estimate within training fold)
oob_ice, oob_melt, oob_avg = cm_errors(confusion_matrix(y_train, rf.oob_decision_function_.argmax(axis=1)))
oob_avg = 1 - rf.oob_score_  # use sklearn's value directly for avg

# Val error (held-out 20%)
val_ice, val_melt, val_avg = cm_errors(confusion_matrix(y_val, rf.predict(X_val)))

print(f'\nTrained in {time.time()-t0:.1f}s')
print(f'\n{"":14s} {"Ice err":>8} {"Melt err":>9} {"Avg err":>8}')
print(f'  OOB          {oob_ice:>8.4f} {oob_melt:>9.4f} {oob_avg:>8.4f}')
print(f'  Val          {val_ice:>8.4f} {val_melt:>9.4f} {val_avg:>8.4f}')

# Show top-20 importances
print('\nFeature importances (top 20):')
top_feats = sorted(zip(FEATURE_COLS, rf.feature_importances_), key=lambda x: -x[1])[:20]
for feat, imp in top_feats:
    bar = '█' * int(imp * 200)
    print(f'  {feat:15s} {imp:.4f}  {bar}')

Features (4): ['elevation', 'edge_distance', 'aspect', 'slope']
Train: 400,000  Val: 100,000
Training...

Trained in 110.6s

                Ice err  Melt err  Avg err
  OOB            0.1545    0.1282   0.1413
  Val            0.1546    0.1257   0.1402

Feature importances (top 20):
  edge_distance   0.6459  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  elevation       0.1630  ████████████████████████████████
  aspect          0.1114  ██████████████████████
  slope           0.0797  ███████████████


In [6]:
# SAVE RF MODEL IF DESIRED
FILE_NAME = "RF_EASD_NOSTRAT"
joblib.dump(rf, LOCAL_DIR / f'Saved_Models\\RFs\\{FILE_NAME}.joblib')

['C:\\Users\\admin\\Documents\\Glacier Project\\Saved_Models\\RFs\\RF_EASD_NOSTRAT.joblib']

In [5]:
# MLP MODEL TRAINING + EVALUATION

scaler_ae = StandardScaler()
X_train_scaled = scaler_ae.fit_transform(X_train)
X_val_scaled = scaler_ae.transform(X_val)

# Train MLP
print('\nTraining MLP...')
t0 = time.time()
mlp_ae = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    alpha=0.0001,
    batch_size=512,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=False
)
mlp_ae.fit(X_train_scaled, y_train)
print(f'Trained in {time.time()-t0:.1f}s ({mlp_ae.n_iter_} epochs)')

y_pred = mlp_ae.predict(X_val_scaled)
cm = confusion_matrix(y_val, y_pred)
ice_err = 1 - cm[0,0]/cm[0].sum()
melt_err = 1 - cm[1,1]/cm[1].sum()
avg_err = 1 - (cm[0,0] + cm[1,1])/cm.sum()
print(f'Validation: Ice {ice_err:.4f}, Melt {melt_err:.4f}, Avg {avg_err:.4f}')


Training AE-edge MLP...
Trained in 141.1s (36 epochs)
Validation: Ice 0.1325, Melt 0.1370, Avg 0.1346


In [6]:
# SAVE MLP MODEL + SCALER IF DESIRED
FILE_NAME = "MLP_AE64_ELE_EDGE_STRAT"
joblib.dump(mlp_ae, LOCAL_DIR / f'Saved_Models\\MLPs\\{FILE_NAME}.joblib')
joblib.dump(scaler_ae, LOCAL_DIR / f'Saved_Models\\MLPs\\{FILE_NAME}_SCALER.joblib')


['C:\\Users\\admin\\Documents\\Glacier Project\\Saved_Models\\MLPs\\MLP_AE64_ELE_EDGE_STRAT_SCALER.joblib']